## Loading Data and Preprocessing

In [2]:
import pandas as pd
import requests
import time

# =========================
# 1. CONFIG
# =========================
symbols = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN']
my_key = 'ZEHOGYARNOL42DO2'
base_url = 'https://www.alphavantage.co/query'
all_stocks = []

# =========================
# 2. HELPERS
# =========================
def call_api(params, pause=15):
    r = requests.get(base_url, params=params)
    data = r.json()
    time.sleep(pause)  # stay safe with free tier
    return data

def safe_float(x):
    try:
        return float(x)
    except:
        return None

def get_daily_data(symbol):
    params = {
        'function': 'TIME_SERIES_DAILY',
        'symbol': symbol,
        'apikey': my_key
    }
    data = call_api(params)

    if 'Time Series (Daily)' not in data:
        print(f"TIME_SERIES_DAILY failed for {symbol}: {data}")
        return None

    df = pd.DataFrame(data['Time Series (Daily)']).T
    df.columns = ['open', 'high', 'low', 'close', 'volume']
    df = df.astype(float)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df['symbol'] = symbol
    return df

def get_overview(symbol):
    params = {
        'function': 'OVERVIEW',
        'symbol': symbol,
        'apikey': my_key
    }
    data = call_api(params)

    if not data or 'Symbol' not in data:
        print(f"OVERVIEW failed for {symbol}: {data}")
        return {}

    return {
        'MarketCapitalization': safe_float(data.get('MarketCapitalization')),
        'PERatio': safe_float(data.get('PERatio')),
        'ProfitMargin': safe_float(data.get('ProfitMargin')),
        'OperatingMarginTTM': safe_float(data.get('OperatingMarginTTM'))
    }

def get_earnings(symbol):
    params = {
        'function': 'EARNINGS',
        'symbol': symbol,
        'apikey': my_key
    }
    data = call_api(params)

    if 'quarterlyEarnings' not in data:
        print(f"EARNINGS failed for {symbol}: {data}")
        return None

    earnings = pd.DataFrame(data['quarterlyEarnings'])
    if earnings.empty:
        return None

    earnings['fiscalDateEnding'] = pd.to_datetime(earnings['fiscalDateEnding'])
    earnings = earnings.sort_values('fiscalDateEnding')

    keep_cols = ['fiscalDateEnding', 'reportedEPS', 'estimatedEPS', 'surprise', 'surprisePercentage']
    earnings = earnings[[c for c in keep_cols if c in earnings.columns]]

    for col in ['reportedEPS', 'estimatedEPS', 'surprise', 'surprisePercentage']:
        if col in earnings.columns:
            earnings[col] = pd.to_numeric(earnings[col], errors='coerce')

    earnings = earnings.rename(columns={'fiscalDateEnding': 'report_date'})
    return earnings

# =========================
# 3. MAIN LOOP
# =========================
for s in symbols:
    print(f"\nFetching data for: {s}")

    # --- API data ---
    df = get_daily_data(s)
    if df is None:
        continue

    overview = get_overview(s)
    earnings = get_earnings(s)

    # =========================
    # 4. LOCAL TECHNICAL FEATURES
    # =========================

    # RSI
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / loss)))

    # Bollinger Bands
    df['MA20'] = df['close'].rolling(window=20).mean()
    df['std'] = df['close'].rolling(window=20).std()
    df['Upper_Band'] = df['MA20'] + (df['std'] * 2)
    df['Lower_Band'] = df['MA20'] - (df['std'] * 2)

    # SMA and EMA
    df['SMA20'] = df['close'].rolling(window=20).mean()
    df['EMA20'] = df['close'].ewm(span=20, adjust=False).mean()

    # MACD
    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

    # =========================
    # 5. TARGET
    # =========================
    df['future_low'] = df['low'].shift(-1).rolling(window=5).min()
    df['downside_risk'] = (df['future_low'] - df['close']) / df['close']

    # =========================
    # 6. ADD OVERVIEW FEATURES
    # =========================
    for k, v in overview.items():
        df[k] = v

    # =========================
    # 7. MERGE EARNINGS ONTO DAILY DATA
    # =========================
    if earnings is not None and not earnings.empty:
        df = df.reset_index().rename(columns={'index': 'date'})
        df = pd.merge_asof(
            df.sort_values('date'),
            earnings.sort_values('report_date'),
            left_on='date',
            right_on='report_date',
            direction='backward'
        )
        df = df.set_index('date')

    # save this stock's finished data
    all_stocks.append(df)

# =========================
# 8. FINAL DATASET
# =========================
master_df = pd.concat(all_stocks)

# remove rows where rolling indicators / future target are missing
master_df = master_df.dropna(subset=[
    'RSI', 'MA20', 'Upper_Band', 'Lower_Band',
    'SMA20', 'EMA20', 'MACD', 'MACD_signal',
    'downside_risk'
])

master_df.to_csv('Master_Stock_Dataset_Expanded.csv')
print("\nDone! Saved as Master_Stock_Dataset_Expanded.csv")
print(master_df.head())
print(master_df.shape)


Fetching data for: AAPL
TIME_SERIES_DAILY failed for AAPL: {'Information': 'We have detected your API key as ZEHOGYARNOL42DO2 and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.'}

Fetching data for: MSFT
TIME_SERIES_DAILY failed for MSFT: {'Information': 'We have detected your API key as ZEHOGYARNOL42DO2 and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.'}

Fetching data for: NVDA


KeyboardInterrupt: 

## Visuals

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df['downside_risk'], bins=50)
plt.title("Distribution of Downside Risk")
plt.xlabel("Downside Risk")
plt.ylabel("Frequency")
plt.show()

In [3]:
df_sorted = df.sort_values("date")

plt.plot(df_sorted['date'], df_sorted['downside_risk'])
plt.title("Downside Risk Over Time")
plt.xlabel("Date")
plt.ylabel("Downside Risk")
plt.show()

AttributeError: 'NoneType' object has no attribute 'sort_values'

In [4]:
plt.scatter(df['RSI'], df['downside_risk'], alpha=0.3)
plt.title("RSI vs Downside Risk")
plt.xlabel("RSI")
plt.ylabel("Downside Risk")
plt.show()

NameError: name 'plt' is not defined

In [5]:
df['band_width'] = df['Upper_Band'] - df['Lower_Band']

plt.scatter(df['band_width'], df['downside_risk'], alpha=0.3)
plt.title("Volatility vs Downside Risk")
plt.xlabel("Bollinger Band Width")
plt.ylabel("Downside Risk")
plt.show()

TypeError: 'NoneType' object is not subscriptable

In [7]:
import numpy as np
plt.scatter(np.log1p(df['volume']), df['downside_risk'], alpha=0.3)
plt.title("Volume vs Downside Risk")
plt.xlabel("Volume")
plt.ylabel("Downside Risk")
plt.show()

NameError: name 'plt' is not defined

In [8]:
plt.scatter(df['MACD'], df['downside_risk'], alpha=0.3)
plt.title("MACD vs Downside Risk")
plt.xlabel("MACD")
plt.ylabel("Downside Risk")
plt.show()

NameError: name 'plt' is not defined

In [9]:
import seaborn as sns

corr = df.select_dtypes(include='number').corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

AttributeError: 'NoneType' object has no attribute 'select_dtypes'